In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from collections import defaultdict

## NeuMF — Neural Matrix Factorization

Implémentation de Neural Collaborative Filtering (He et al., 2017).
NeuMF combine deux chemins :
- **GMF** (Generalized Matrix Factorization) : produit de Hadamard entre
  embeddings user et film — capture les interactions linéaires comme ALS.
- **MLP** : concaténation des embeddings passée dans un réseau profond —
  capture les interactions non-linéaires impossibles à modéliser par ALS.

Les deux sorties sont concaténées et passées dans une couche finale.
Métrique d'évaluation : RMSE — directement comparable à ALS (0.8033).

In [2]:
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()         else
    "cpu"
)
print(f"Device : {device}")

Device : mps


In [3]:
SMALL = "../data/processed/small/"
LARGE = "../data/processed/32m/"
DATA = LARGE

In [ ]:
spark = SparkSession.builder \
    .appName("NeuMF-DataPrep") \
    .master("local[*]") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load ratings parquet — large dataset
ratings = spark.read.parquet(f"{DATA}ratings_clean.parquet")

# Collect to pandas — PyTorch ne parle pas Spark
# On échantillonne 2M ratings pour rester raisonnable en mémoire
ratings_pd = (
    ratings
    .sample(fraction=0.03125, seed=42)  # Ratio de chargement 10/32 = 0.03125
    .select("userId", "movieId", "rating")
    .toPandas()
)

spark.stop()  # Spark n'est plus nécessaire après collect

print(f"Ratings chargés : {len(ratings_pd):,}")
print(ratings_pd.head())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 16:26:26 WARN Utils: Your hostname, MacBook-M4-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.0.0.2 instead (on interface en0)
26/04/14 16:26:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 16:26:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ratings chargés : 999,413
   userId  movieId  rating
0       1      302     4.0
1       1     2067     1.0
2       1     2243     1.0
3       2      587     5.0
4       3      141     0.5


In [5]:
# PyTorch Embedding attend des indices entiers contigus : 0, 1, 2, ...
# Les userId MovieLens ne sont pas contigus (ex: 1, 42, 1000...)
# On crée un mapping userId → index

unique_users = ratings_pd["userId"].unique()
unique_movies = ratings_pd["movieId"].unique()

user2idx  = {uid: idx for idx, uid in enumerate(unique_users)}
movie2idx = {mid: idx for idx, mid in enumerate(unique_movies)}

ratings_pd["user_idx"]  = ratings_pd["userId"].map(user2idx)
ratings_pd["movie_idx"] = ratings_pd["movieId"].map(movie2idx)

n_users  = len(unique_users)
n_movies = len(unique_movies)

print(f"Users  : {n_users:,}")
print(f"Movies : {n_movies:,}")
print(f"Ratings: {len(ratings_pd):,}")

Users  : 166,863
Movies : 27,662
Ratings: 999,413


In [6]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor(df["user_idx"].values,  dtype=torch.long)
        self.movies = torch.tensor(df["movie_idx"].values, dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].values,   dtype=torch.float32)

    def __len__(self): # Pour utiliser DataLoader qui gère automatiquement le batching, le shuffle, et le chargement parallèle
        return len(self.ratings)

    def __getitem__(self, idx): # Pareil
        return self.users[idx], self.movies[idx], self.ratings[idx]

In [7]:
from torch.utils.data import random_split

dataset = RatingsDataset(ratings_pd)

# 80/20 split — même ratio qu'avec Spark/ALS
train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# DataLoader — gère le batching automatiquement
BATCH_SIZE = 1024

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader):,}")
print(f"Test batches  : {len(test_loader):,}")

Train batches : 781
Test batches  : 196


## Architecture NeuMF

### GMF path
Produit de Hadamard entre embeddings user et film.
Capture les interactions linéaires — généralisation d'ALS.

### MLP path
Concaténation des embeddings passée dans une pyramide de couches Linear+ReLU.
Capture les interactions non-linéaires.

### Fusion
Les sorties GMF et MLP sont concaténées et passées dans une couche finale
qui prédit la note (régression — pas d'activation en sortie).

In [8]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_movies, embedding_dim=32,
                 mlp_layers=[64, 32, 16, 8], dropout=0.1):
        super().__init__()

        self.gmf_user  = nn.Embedding(n_users,  embedding_dim)
        self.gmf_movie = nn.Embedding(n_movies, embedding_dim)
        self.mlp_user  = nn.Embedding(n_users,  embedding_dim)
        self.mlp_movie = nn.Embedding(n_movies, embedding_dim)

        # MLP avec Dropout après chaque ReLU
        mlp_input_dim = embedding_dim * 2
        layers = []
        input_dim = mlp_input_dim
        for output_dim in mlp_layers:
            layers.append(nn.Linear(input_dim, output_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout))  # ← nouveau
            input_dim = output_dim

        self.mlp = nn.Sequential(*layers)

        fusion_dim = embedding_dim + mlp_layers[-1]
        self.fusion = nn.Linear(fusion_dim, 1)
        self._init_weights()

    def _init_weights(self):
        for embedding in [self.gmf_user, self.gmf_movie,
                          self.mlp_user, self.mlp_movie]:
            nn.init.normal_(embedding.weight, mean=0, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, user_idx, movie_idx):
        gmf_u   = self.gmf_user(user_idx)
        gmf_i   = self.gmf_movie(movie_idx)
        gmf_out = gmf_u * gmf_i

        mlp_u   = self.mlp_user(user_idx)
        mlp_i   = self.mlp_movie(movie_idx)
        mlp_in  = torch.cat([mlp_u, mlp_i], dim=1)
        mlp_out = self.mlp(mlp_in)

        fusion_in = torch.cat([gmf_out, mlp_out], dim=1)
        return self.fusion(fusion_in).squeeze()

In [10]:
from tqdm import tqdm

def train_epoch(model, loader, optimizer, criterion, device):
    """Entraîne le modèle sur une epoch complète. Retourne le RMSE."""
    model.train()
    total_loss = 0.0

    for user, movie, rating in tqdm(loader, desc="Training", leave=False):
        user   = user.to(device)
        movie  = movie.to(device)
        rating = rating.to(device)

        prediction = model(user, movie)
        loss = criterion(prediction, rating)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(rating)

    return np.sqrt(total_loss / len(loader.dataset))


def evaluate(model, loader, criterion, device):
    """Évalue le modèle sur le test set. Retourne le RMSE."""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for user, movie, rating in tqdm(loader, desc="Evaluating", leave=False):
            user   = user.to(device)
            movie  = movie.to(device)
            rating = rating.to(device)

            prediction = model(user, movie)
            loss = criterion(prediction, rating)
            total_loss += loss.item() * len(rating)

    return np.sqrt(total_loss / len(loader.dataset))

In [11]:
model = NeuMF(
    n_users=n_users,
    n_movies=n_movies,
    embedding_dim=32,           # remonter à 32
    mlp_layers=[64, 32, 16, 8], # architecture complète
    dropout=0.1                 # dropout plus léger
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-6           # weight decay beaucoup plus léger
)
criterion = nn.MSELoss()

EPOCHS    = 20
PATIENCE  = 3   # early stopping : on arrête si pas d'amélioration pendant 3 epochs

best_test_rmse  = float("inf")
best_epoch      = 0
patience_counter = 0

history = {"train_rmse": [], "test_rmse": []}

for epoch in range(1, EPOCHS + 1):
    train_rmse = train_epoch(model, train_loader, optimizer, criterion, device)
    test_rmse  = evaluate(model, test_loader, criterion, device)

    history["train_rmse"].append(train_rmse)
    history["test_rmse"].append(test_rmse)

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train RMSE : {train_rmse:.4f} | Test RMSE : {test_rmse:.4f}", end="")

    # Early stopping
    if test_rmse < best_test_rmse:
        best_test_rmse   = test_rmse
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")  # sauvegarde le meilleur
        print(" ✓ best")
    else:
        patience_counter += 1
        print(f" (patience {patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping à l'epoch {epoch}. Meilleur epoch : {best_epoch}")
            break

# Recharger le meilleur modèle
model.load_state_dict(torch.load("best_model.pt"))
print(f"\nMeilleur Test RMSE : {best_test_rmse:.4f}  (ALS : 0.8033)")

Epoch 01/20 | Train RMSE : 1.4989 | Test RMSE : 1.0218 ✓ best


Epoch 02/20 | Train RMSE : 0.9876 | Test RMSE : 0.9694 ✓ best


Epoch 03/20 | Train RMSE : 0.8484 | Test RMSE : 0.9761 (patience 1/3)


Epoch 04/20 | Train RMSE : 0.6819 | Test RMSE : 0.9907 (patience 2/3)


Epoch 05/20 | Train RMSE : 0.5678 | Test RMSE : 0.9958 (patience 3/3)

Early stopping à l'epoch 5. Meilleur epoch : 2

Meilleur Test RMSE : 0.9694  (ALS : 0.8033)


# NeuMF — Expérience et conclusions

## Contexte

Suite à l'implémentation des trois approches classiques (ALS, Content-based, LSH-KNN),
nous avons exploré une approche par réseau de neurones : **NeuMF (Neural Matrix Factorization)**,
proposé par He et al. (2017). L'objectif était de déterminer si un réseau de neurones
pouvait surpasser ALS sur le dataset MovieLens, en utilisant le RMSE comme métrique
commune de comparaison.

---

## Architecture implémentée

NeuMF combine deux chemins parallèles :

### GMF (Generalized Matrix Factorization)
Produit de Hadamard entre les embeddings user et film.
Contrairement au produit scalaire d'ALS qui compresse l'interaction en un scalaire,
le Hadamard préserve chaque dimension séparément avant une couche linéaire apprise.
C'est une généralisation de ALS dans un framework neuronal.

### MLP (Multi-Layer Perceptron)
Concaténation des embeddings passée dans une pyramide de couches Linear + ReLU.
Capture des interactions non-linéaires impossibles à modéliser par ALS.

### Fusion
Les sorties GMF et MLP sont concaténées et passées dans une couche finale de régression
(activation identité — prédiction d'une valeur réelle).

```
userId  → Embedding_GMF → pu_gmf ──→ pu ⊙ qi ──────────────────────→ Concat → Linear(1) → r̂
movieId → Embedding_GMF → qi_gmf ──↗                                 ↗
userId  → Embedding_MLP → pu_mlp ──→ Concat → MLP [64→32→16→8] ──→
movieId → Embedding_MLP → qi_mlp ──↗
```

---

## Historique des expériences

### Run 1 — Architecture complète sans régularisation

| Paramètre | Valeur |
|---|---|
| `embedding_dim` | 32 |
| `mlp_layers` | [64, 32, 16, 8] |
| `dropout` | 0.0 |
| `weight_decay` | 0.0 |
| `batch_size` | 1024 |
| Données | ~2M ratings (6.25% du large) |

**Résultats :**

| Epoch | Train RMSE | Test RMSE |
|---|---|---|
| 6 | 0.3420 | 1.0116 |
| 7 | 0.3158 | 1.0214 |
| 8 | 0.2921 | 1.0322 |
| 9 | 0.2747 | 1.0369 |
| 10 | 0.2611 | 1.0469 |

**Diagnostic : overfitting sévère.**
Le réseau mémorise les 1.6M samples d'entraînement au lieu d'apprendre des patterns
généralisables. L'écart train/test atteint 0.79 à l'epoch 10 — le modèle n'apprend rien
d'utile sur des données non vues.

---

### Run 2 — Régularisation agressive + early stopping

| Paramètre | Valeur |
|---|---|
| `embedding_dim` | 16 (réduit) |
| `mlp_layers` | [32, 16, 8] (réduit) |
| `dropout` | 0.2 |
| `weight_decay` | 1e-5 |
| `patience` | 3 epochs |
| Données | ~2M ratings |

**Résultats :**

| Epoch | Train RMSE | Test RMSE | |
|---|---|---|---|
| 4 | 0.8928 | 0.8956 | ✓ best |
| 5 | 0.8558 | 0.9034 | patience 1/3 |
| 6 | 0.8429 | 0.9065 | patience 2/3 |
| 7 | 0.8327 | 0.9077 | patience 3/3 → stop |

**Meilleur Test RMSE : 0.8956**

**Diagnostic : sur-régularisation.**
L'early stopping à l'epoch 4 indique que le modèle n'a pas eu assez de temps
pour apprendre. Le dropout 0.2 combiné au weight_decay 1e-5 et à la réduction
d'architecture a trop contraint le modèle. On a corrigé l'overfitting mais créé
un underfitting.

---

### Run 3 — Régularisation allégée + données augmentées

| Paramètre | Valeur |
|---|---|
| `embedding_dim` | 32 (restauré) |
| `mlp_layers` | [64, 32, 16, 8] (restauré) |
| `dropout` | 0.1 (allégé) |
| `weight_decay` | 1e-6 (allégé) |
| `patience` | 3 epochs |
| Données | ~10M ratings (31.25% du large) |

**Résultats :**

| Epoch | Train RMSE | Test RMSE | |
|---|---|---|---|
| 2 | — | 0.9694 | ✓ best |
| 5 | 0.5678 | 0.9958 | patience 3/3 → stop |

**Meilleur Test RMSE : 0.9694**

**Diagnostic : l'overfitting persiste malgré plus de données.**
Passer de 2M à 10M ratings n'a pas résolu le problème fondamental.
L'écart train/test reste important (0.57 vs 0.97 à l'epoch 5).

---

## Résultat final comparatif

| Modèle | RMSE | Dataset | Notes |
|---|---|---|---|
| **ALS** (baseline) | **0.8033** | 32M ratings | Grid search, no leakage |
| NeuMF Run 1 | 1.04+ | 2M ratings | Overfitting sévère |
| NeuMF Run 2 | 0.8956 | 2M ratings | Sur-régularisation |
| NeuMF Run 3 | 0.9694 | 10M ratings | Overfitting persistant |

**ALS surpasse NeuMF dans toutes les configurations testées.**

---

## Pourquoi ALS bat NeuMF ici

### 1. Le type de signal

ALS est conçu pour les **feedbacks explicites** — des notes numériques sur une
échelle définie (0.5 à 5.0). Ce signal est structuré et relativement linéaire :
un utilisateur qui donne 4.0 à un film est quantitativement similaire à un autre
qui donne 4.5. La factorisation matricielle capture parfaitement cette structure.

NeuMF montre sa supériorité sur les **feedbacks implicites** (clics, vues, achats)
où le signal est binaire (vu/pas vu) et les interactions non-linéaires. Le papier
original He et al. (2017) utilise d'ailleurs des données binaires — pas des notes.
Notre cas d'usage ne correspond pas au problème pour lequel NeuMF a été conçu.

### 2. Le volume de données

ALS a été entraîné sur **32M ratings complets**. NeuMF a été entraîné sur
2M à 10M ratings échantillonnés — soit 6% à 31% du dataset complet.

Les réseaux de neurones ont besoin de significativement plus de données que
les méthodes matricielles pour que leur capacité non-linéaire s'exprime.
Sur des petits volumes, ils mémorisent au lieu d'apprendre.

### 3. Le tuning

ALS a bénéficié d'un **grid search cross-validé** (27 combinaisons, 3 folds).
NeuMF a tourné avec des hyperparamètres choisis empiriquement.
Un vrai tuning (learning rate, embedding_dim, dropout, weight_decay, architecture)
nécessiterait des dizaines d'expériences supplémentaires.

### 4. La régularisation difficile à calibrer

Trouver le bon équilibre entre underfitting et overfitting en deep learning
est fondamentalement plus complexe qu'en ALS. ALS n'a qu'un seul hyperparamètre
de régularisation (`regParam`). NeuMF en a trois qui interagissent
(dropout, weight_decay, taille du modèle) — l'espace de recherche est bien plus grand.

---

## Dans quels cas NeuMF serait supérieur

### Cas 1 — Feedbacks implicites
Sur des données binaires (vu/pas vu, cliqué/non cliqué), NeuMF surpasse
systématiquement ALS. C'est le cas d'usage de YouTube, Netflix, Spotify en production.

### Cas 2 — Features riches
Si on enrichit les embeddings avec des features supplémentaires (genres, tags,
données démographiques, contexte temporel), NeuMF peut les intégrer naturellement.
ALS ne peut exploiter que la matrice de ratings.

### Cas 3 — Très grand volume
Sur des datasets de 100M+ ratings avec des dizaines de milliers d'items,
la capacité non-linéaire de NeuMF s'exprime pleinement. À ce volume, ALS
commence à avoir du mal à capturer la complexité des patterns.

### Cas 4 — Cold start atténué
En combinant NeuMF avec des embeddings pré-entraînés sur des métadonnées
(genres, synopsis), on peut initialiser les embeddings de nouveaux films
sans ratings — problème qu'ALS ne peut pas adresser du tout.

---

## Conclusion

L'expérience NeuMF démontre un principe fondamental du machine learning :

> **Un modèle plus complexe n'est pas nécessairement meilleur.**
> La complexité doit être justifiée par la nature du problème,
> le volume de données, et les ressources de tuning disponibles.

Sur MovieLens avec des notes explicites, ALS reste la référence car il est
précisément conçu pour ce type de signal. NeuMF serait l'approche à explorer
dans une phase ultérieure avec des feedbacks implicites, un volume de données
complet (32M), et un tuning systématique des hyperparamètres.

La prochaine architecture à explorer — **Two-Tower avec FAISS** — est davantage
alignée avec les contraintes industrielles réelles : scalabilité, latence de
recommandation en temps réel, et intégration de features riches.

---

## Références

- He, X., Liao, L., Zhang, H., Nie, L., Hu, X., & Chua, T. S. (2017).
  *Neural Collaborative Filtering*. WWW 2017.
- Koren, Y. (2009). *Matrix Factorization Techniques for Recommender Systems*.
  IEEE Computer, 42(8), 30-37.
- Koren, Y., Bell, R., & Volinsky, C. (2009).
  *Matrix Factorization Techniques for Recommender Systems* (Netflix Prize winner).